# FASE 2. LIMPIEZA DE DATOS - ESPIRAS Y COORDENADAS

## 1. Seteo de la Sesion Spark

In [1]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import Window

spark = SparkSession.builder \
    .appName("TFM_Espiras_Ubicaciones_Coordenadas") \
    .getOrCreate()

:: loading settings :: url = jar:file:/usr/lib/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-cc65ba22-87b0-4245-ac73-b8324a17fa4e;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.3.2 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.3.2 in central
	found org.apache.kafka#kafka-clients;2.8.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.8.4 in central
	found org.slf4j#slf4j-api;1.7.32 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.2 in central
	found org.spark-project.spark#unused;1.0.0 in central
	found org.apache.hadoop#hadoop-client-api;3.3.2 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report 

**Decisiones:**
- Se excluye la espira 829 (quedan 964 espiras).
- Se excluirán las 18 espiras fuera de Valencia.
- Para cada espira se usa la ubicación válida con más registros, que es la vigente desde el 16 de octubre de 2023.

## 2. LIMPIEZA

### 2.1 Excluir la espira 829 y las ubicaciones no válidas

In [4]:
espiras_limpias = (spark.read.parquet("gs://tfm_unir1/processed/trafico/espiras/")
            .select("id_espira").distinct())

print("Espiras limpias:", espiras_limpias.count())

Espiras limpias: 965


In [6]:
archivo_trafico_inicial = spark.read.parquet("gs://tfm_unir1/trafico/espires_VLC2_unificado")

ubicaciones = (archivo_trafico_inicial
    .select(F.col("detector_id").alias("id_espira"), "latitude", "longitude", "snapshot_datetime")
    .join(espiras_limpias, "id_espira")
    .groupBy("id_espira", "latitude", "longitude")
    .agg(
        F.count("*").alias("registros"),
        F.min("snapshot_datetime").alias("desde"),
        F.max("snapshot_datetime").alias("hasta")))

ubicaciones.cache()

DataFrame[id_espira: string, latitude: double, longitude: double, registros: bigint, desde: timestamp, hasta: timestamp]

In [8]:
ubicaciones = ubicaciones.withColumn("valida",
    F.coalesce(
        F.col("latitude").between(39.27, 39.57) & F.col("longitude").between(-0.45, -0.27),
        F.lit(False)))

In [9]:
ubicaciones_limpias = ubicaciones.filter((F.col("id_espira") != "829") & F.col("valida"))

print("Espiras:", ubicaciones_limpias.select("id_espira").distinct().count())   # 964

Espiras: 964


### 2.2 Una ubicación por espira: la válida con más registros

In [10]:
orden = Window.partitionBy("id_espira").orderBy(F.desc("registros"))

ubicacion_final = (ubicaciones_limpias
    .withColumn("puesto", F.row_number().over(orden))
    .filter(F.col("puesto") == 1)
    .select("id_espira", "latitude", "longitude"))

print("Espiras con ubicación:", ubicacion_final.count())   # 964
ubicacion_final.show(5)

Espiras con ubicación: 964
+---------+------------------+-------------------+
|id_espira|          latitude|          longitude|
+---------+------------------+-------------------+
|     1436| 39.47575341289948|-0.3542245583258239|
|     1512| 39.47748989535151|-0.3516479908309456|
|     2904|39.471830270124485|-0.3273659943075868|
|     3210| 39.47721545857593|-0.3247574126588342|
|     3606|  39.4743687713479|-0.4060560377175244|
+---------+------------------+-------------------+
only showing top 5 rows



### 2.3 Exportar para QGIS

Se guarda un CSV con `id_espira`, `latitude` y `longitude`. En QGIS se asigna el distrito con una unión espacial (*Unir atributos por localización*) contra el GeoJSON de distritos.

In [11]:
ubicacion_final.coalesce(1).write.mode("overwrite").option("header", True) \
    .csv("gs://tfm_unir1/processed/espira_distrito/")

print("Exportado")

Exportado


### 2.4 Exportar a Parquet
Una vez obtenido el distrito usando QGIS, guardamos en la silver el parquet


In [17]:
from pyspark.sql import functions as F

dim_espira = (
    spark.read.option("header", True)
    .csv("gs://tfm_unir1/espira_distrito.csv")
    .select(
        F.col("id_espira").cast("string").alias("id_espira"),
        F.col("latitude").cast("double").alias("latitud"),
        F.col("longitude").cast("double").alias("longitud"),
        F.col("coddistrit").cast("int").alias("id_distrito"),
    )
)

# Comprobaciones
dim_espira.agg(
    F.count("*").alias("espiras"),                                        # 964
    F.countDistinct("id_espira").alias("espiras_distintas"),              # 964
    F.sum(F.col("id_distrito").isNull().cast("int")).alias("sin_distrito")  # 0
).show()

dim_espira.write.mode("overwrite").parquet("gs://tfm_unir1/processed/espira_distrito/")
print("Guardado")

+-------+-----------------+------------+
|espiras|espiras_distintas|sin_distrito|
+-------+-----------------+------------+
|    964|              964|           0|
+-------+-----------------+------------+



Guardado


In [18]:
#Estaciones
(spark.read.parquet("gs://tfm_unir1/processed/estaciones/")
    .select("id_estacion", "nombre_estacion", "latitud", "longitud")
    .coalesce(1)
    .write.mode("overwrite").option("header", True)
    .csv("gs://tfm_unir1/estaciones/"))